<a href="https://colab.research.google.com/github/niharnandala/ntsb-safety-review-agent/blob/main/notebooks/03_tools_and_rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · The agent's tools

Phase 2. In `02` I built the episodes: a case frozen on the day a reply arrives, with NTSB's real answer
sealed inside. The AI can't just get the whole episode pasted in:
- the `answer` is right there in the JSON, so it would just copy it
- some cases are huge (up to 107 letters), and Tiny Aya only holds ~8,000 tokens (~30,000 characters)
- a real analyst compares with older cases, and those are in 20,000 other episodes

So I give the AI **buttons (tools)**. Each one fetches one small piece, and none of them can reach the sealed
answer. The AI never runs code or opens files itself. It asks for a tool in text, our code runs it, and the
result goes back to it as text.

**What's in here, in the order I built it:**

| # | Piece | What it does |
|---|---|---|
| 0 | desk + `run_tool` + audit log | the open case with the answer removed, the one door for every tool call, and the logbook |
| 1 | `get_recommendation` | the case summary the AI reads first |
| 2 | `read_letter` | one letter, 4,000 characters at a time (our letters and peer letters) |
| 3 | `search_history` | find which letters in this case mention some words |
| 4 | `find_precedents` | similar past cases + the grade NTSB gave them (RAG, BM25) |
| 5 | `get_peers` | how NTSB handled the other orgs that got the same recommendation |
| 6 | `verify_quote` | is the AI's quote really in the letter? |
| 7 | `days_late` (helper) | how many days past NTSB's 90-day window the reply came |
| 8 | `submit_review` → `check_rules` → `request_approval` | draft → rulebook → human queue |
| 9 | `read_attachment`, `record_decision` | honest attachment stub, and the human-only approve/reject |

Then the tool menu (what Tiny Aya reads) and a safety check on 50 real cases.

I test every tool on the same real case: West Virginia's reply about ferry safety (recommendation M-05-007,
shown as CASE-7360). **Runtime:** CPU is enough.

## Setup
Mount Drive (our episodes from `02` live there) and make a folder for the audit logs.

In [1]:
!pip install -q rank_bm25

from google.colab import drive
drive.mount("/content/drive")

import os
DATA = "/content/drive/MyDrive/ntsb_agent/data"
LOGS = "/content/drive/MyDrive/ntsb_agent/logs"
os.makedirs(LOGS, exist_ok=True)

Mounted at /content/drive


## Load the episodes
Each file has one episode per line (`.jsonl`), so I read it line by line and turn each line into a dict.

In [2]:
import json, re
from datetime import datetime, timezone

def load(split):
    episodes = []
    with open(f"{DATA}/episodes_{split}.jsonl") as f:
        for line in f:
            episodes.append(json.loads(line))
    return episodes

train = load("train")
val = load("val")
test = load("test")
print(len(train), "train |", len(val), "val |", len(test), "test")

20609 train | 300 val | 300 test


## Settings
The numbers we agreed on from the letter lengths in `02`, all in one place:
- `read_letter` gives **4,000 characters** per page (90%+ of replies fit in one page)
- `search_history` shows **max 20** previews of ~**150 characters**
- the drafted letter should stay under **~1,900 characters**
- confidence below **0.6** → a human should look closer
- NTSB asks for a reply within **90 days**

And the 7 grades. Nothing else is allowed.

In [3]:
PAGE_SIZE = 4000
MAX_HITS = 20
PREVIEW = 150
LETTER_LIMIT = 1900
LOW_CONFIDENCE = 0.6
REPLY_WINDOW = 90

GRADES = [
    "Open – Acceptable",
    "Open – Acceptable Alternate",
    "Open – Unacceptable",
    "Closed – Acceptable",
    "Closed – Acceptable Alternate",
    "Closed – Unacceptable",
    "Closed – Other",
]

## The grade finder (from `02`)
`get_recommendation` tells the AI "the last grade NTSB gave in this case was X". That grade comes from
NTSB's **old** letters, which are already in the visible history, so it's fair to show. For that I need
the two helpers from `02`:

- `find_grades(text)`: finds every grade phrase in a letter, as `(position, grade words)`.
  It reads both styles: `"Open--Acceptable Response."` (in quotes) and `OPEN—ACCEPTABLE RESPONSE` (no quotes)
- `to_class(grade words)`: turns the messy words into one of our 7 grades,
  e.g. `"CLOSED—ACCEPTABLE ALTERNATE ACTION"` → `"Closed – Acceptable Alternate"`

In [4]:
# old style: the grade sits in quotes -> "Open--Acceptable Response."
QUOTED_GRADE = re.compile(r'["“]\s*((?:Open|Closed)\s*[-–—]+\s*[A-Za-z /]+?)\s*[."”,]', re.I)

# new style (mostly 2010+): no quotes -> OPEN—ACCEPTABLE RESPONSE
# without quotes we can't tell where the grade ends, so only NTSB's real grade words count
PLAIN_GRADE = re.compile(
    r"\b((?:OPEN|CLOSED)\s*[-–—]+\s*"
    r"(?:ACCEPTABLE\s+ALTERNATE\s+(?:RESPONSE|ACTION)"
    r"|UNACCEPTABLE\s+(?:RESPONSE|ACTION)(?:\s*/\s*NO\s+RESPONSE\s+RECEIVED)?"
    r"|ACCEPTABLE\s+(?:RESPONSE|ACTION)"
    r"|EXCEEDS\s+RECOMMENDED\s+ACTION|AWAIT\s+RESPONSE|NO\s+LONGER\s+APPLICABLE"
    r"|SUPERSEDED|SUPERCEDED|RECONSIDERED))",
    re.I,
)

def find_grades(text):
    found = []
    quoted_spans = []

    # 1. grades in quotes
    for m in QUOTED_GRADE.finditer(text):
        found.append((m.start(), m.group(1)))
        quoted_spans.append((m.start(), m.end()))

    # 2. grades without quotes (skip the ones we already found inside quotes)
    for m in PLAIN_GRADE.finditer(text):
        inside_quotes = False
        for start, end in quoted_spans:
            if start <= m.start() < end:
                inside_quotes = True
        if not inside_quotes:
            found.append((m.start(), m.group(1)))

    found.sort()          # reading order, by position
    return found

def to_class(grade):
    g = grade.lower()
    if g.startswith("open"):
        side = "Open"
    else:
        side = "Closed"

    if "await" in g:
        return None                          # "await response" isn't a grade on a reply
    if "supersed" in g or "superced" in g:
        return "Closed – Other"
    if "unacceptable" in g:                  # check before "acceptable" (it contains it)
        return side + " – Unacceptable"
    if "alternate" in g:
        return side + " – Acceptable Alternate"
    if "acceptable" in g or "exceed" in g:
        return side + " – Acceptable"
    if side == "Closed":
        return "Closed – Other"              # no longer applicable, reconsidered
    return None

In [5]:
# quick look at what they give back
sample = 'CASE-1 is classified "Open--Acceptable Response." and [OTHER-REC] remains OPEN—UNACCEPTABLE RESPONSE'
print(find_grades(sample))
for position, grade_words in find_grades(sample):
    print(grade_words, "->", to_class(grade_words))

[(21, 'Open--Acceptable Response'), (74, 'OPEN—UNACCEPTABLE RESPONSE')]
Open--Acceptable Response -> Open – Acceptable
OPEN—UNACCEPTABLE RESPONSE -> Open – Unacceptable


## The desk
`Desk` = one open case. When I open an episode, I copy **only the visible part** onto the desk:
- skip `answer` (the sealed NTSB grade + letter)
- skip every key starting with `_` (real ID, real dates)
- do the same inside every letter (each letter has a hidden `_date`)

So the answer is never on the desk, and no tool can leak it, even by accident.

Two hidden notes stay on the desk, only for the tools' own filtering. They're never printed:
- `_srid`: the real recommendation ID, so the precedent search can skip this same recommendation
- `_cutoff`: the real date of the reply, so precedents only come from **before** that date

`current` = the desk that's open right now. The AI only sends a tool name + parameters, so the tools
read the case from `current`.

In [6]:
class Desk:
    def __init__(self, episode):
        # copy the visible part only
        self.case = {}
        for key, value in episode.items():
            if key == "answer" or key.startswith("_"):
                continue
            self.case[key] = value

        # same for every letter: drop the hidden real date
        clean_history = []
        for letter in episode["history"]:
            clean_letter = {}
            for key, value in letter.items():
                if key.startswith("_"):
                    continue
                clean_letter[key] = value
            clean_history.append(clean_letter)
        self.case["history"] = clean_history

        # hidden notes for the tools only
        self._srid = episode["_srid"]
        self._cutoff = episode["_reply_date"]

        # filled later
        self.log = []
        self.draft = None
        self.rules_passed = False
        self.approval = None
        self.peers = {}                 # filled by get_peers: "HI" -> Hawaii's episode

    def letter(self, letter_id):
        wanted = letter_id.strip().upper()          # forgive " l-0003 "
        for letter in self.case["history"]:
            if letter["letter_id"] == wanted:
                return letter
        return None


def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


current = None

def open_case(episode):
    global current
    current = Desk(episode)
    current.log.append({
        "step": 0,
        "time": now(),
        "event": "case opened",
        "case_id": current.case["case_id"],
        "episode_id": current.case["episode_id"],
    })
    return current

## `run_tool`: the one door + the audit log
The AI never calls a tool directly. It asks, and `run_tool`:
1. checks the tool exists (if not, it tells the AI, no crash)
2. runs it with the AI's parameters
3. if the parameters are wrong, it tells the AI, no crash
4. **always** writes one line in the log: step, time, tool, parameters, and what came back

`TOOLS` = the menu of allowed tools. I add each tool to it as I build it.
`save_log` writes the log to Drive: one file per case, one line per step.

In [7]:
TOOLS = {}

def run_tool(name, params):
    if name not in TOOLS:
        result = f"ERROR: there is no tool called {name}."
    else:
        try:
            result = TOOLS[name](**params)          # {"letter_id": "L-0003"} -> letter_id="L-0003"
        except TypeError as err:
            result = f"ERROR: wrong parameters for {name}: {err}"

    current.log.append({
        "step": len(current.log),
        "time": now(),
        "tool": name,
        "params": params,
        "result_chars": len(result),
        "result_preview": result[:300],
    })
    return result


def save_log(desk):
    path = f"{LOGS}/{desk.case['episode_id']}.jsonl"
    with open(path, "w") as f:
        for entry in desk.log:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    return path

In [8]:
# test the desk + run_tool with a fake tool
def ping() -> str:
    """Test tool."""
    return "pong"

TOOLS["ping"] = ping

open_case(val[0])
print(run_tool("ping", {}))
print(run_tool("fly", {}))                    # tool that doesn't exist
print(run_tool("ping", {"x": 1}))             # wrong parameters

hidden_on_desk = False
for key in current.case:
    if key == "answer" or key.startswith("_"):
        hidden_on_desk = True
print("answer or hidden keys on the desk?", hidden_on_desk)

for entry in current.log:
    print(entry)

del TOOLS["ping"]

pong
ERROR: there is no tool called fly.
ERROR: wrong parameters for ping: ping() got an unexpected keyword argument 'x'
answer or hidden keys on the desk? False
{'step': 0, 'time': '2026-11-14T16:37:16+00:00', 'event': 'case opened', 'case_id': 'CASE-0028', 'episode_id': 'CASE-0028-E1'}
{'step': 1, 'time': '2026-11-14T16:37:16+00:00', 'tool': 'ping', 'params': {}, 'result_chars': 4, 'result_preview': 'pong'}
{'step': 2, 'time': '2026-11-14T16:37:16+00:00', 'tool': 'fly', 'params': {}, 'result_chars': 35, 'result_preview': 'ERROR: there is no tool called fly.'}
{'step': 3, 'time': '2026-11-14T16:37:16+00:00', 'tool': 'ping', 'params': {'x': 1}, 'result_chars': 79, 'result_preview': "ERROR: wrong parameters for ping: ping() got an unexpected keyword argument 'x'"}


## My test case
West Virginia's reply about ferry safety: recommendation M-05-007 ("encourage your ferry operators to adopt a
safety management system"), sent to all 50 states. On the desk it's **CASE-7360**, with 3 letters:
L-0001 (WV, day 95) · L-0002 (NTSB, day 150, "Open – Acceptable") · **L-0003 (WV, day 474), the one to grade**.
NTSB's real answer (sealed): Closed – Acceptable.

In [9]:
ferry = None
for e in train + val + test:
    if e["_srid"] == "M-05-007" and e["recipient"] == "WV":
        ferry = e                     # keeps the last one = the later episode (3 letters)
print(ferry["episode_id"], "|", len(ferry["history"]), "letters | to grade:", ferry["to_evaluate"])

CASE-7360-E11 | 3 letters | to grade: L-0003


## Tool 1 · `get_recommendation`
The first thing the AI calls. It only gets a case number, so before reading anything it needs the big picture:
- case ID, mode, priority, most wanted, who it was sent to, the recommendation text
- which letter to review
- **NTSB's latest letter**: that's where NTSB last said what it's waiting for. I first had a separate tool
  that picked "ask" sentences out of it with keywords (`pending`, `please`, `we urge`…), but plain Python can't
  understand meaning, so it was a fake. Pointing the AI at the letter is honest and simpler
- **the last grade NTSB gave in this case** (from old NTSB letters, which are visible, so it's fair)
- a list of all letters: ID · day · from · gap · length. **No letter text.** That's what `read_letter` is for,
  so the summary stays small even for a case with 100+ letters

**A bug I caught:** one NTSB letter can grade several recommendations:
`CASE-7360 is classified "Open--Acceptable." [OTHER-REC] is classified "Closed--Unacceptable."`
Taking the last grade in the letter gives the **other** case's grade. On the desk our case is always its
`CASE-xxxx` and every other ID is `[OTHER-REC]`, so `grades_for_case` keeps a grade only if our case ID is in
the text just before it (back to the previous grade). If the letter has no other recommendation at all, the
grades must be ours. Measured on all 21,209 episodes, the old way showed a wrong or guessed last grade in
**1,598 cases (7.5%)**.

In [10]:
def grades_for_case(text, case_id):
    found = find_grades(text)
    grades = []
    previous_end = 0

    # 1. keep a grade only if our case ID is in the text just before it
    for position, grade_words in found:
        before = text[previous_end:position]
        if case_id in before:
            clean = to_class(grade_words)
            if clean is not None:
                grades.append(clean)
        previous_end = position + len(grade_words)

    # 2. a letter about only our case often has no ID right before the grade
    if len(grades) == 0 and "[OTHER-REC]" not in text:
        for position, grade_words in found:
            clean = to_class(grade_words)
            if clean is not None:
                grades.append(clean)

    return grades


def last_ntsb_grade(history, case_id):
    # newest NTSB letter that grades our case -> its last grade
    for letter in reversed(history):
        if letter["from"] != "NTSB":
            continue
        grades = grades_for_case(letter["text"], case_id)
        if len(grades) > 0:
            return grades[-1], letter["letter_id"]
    return None, None


def letter_line(letter):
    line = f"{letter['letter_id']} · Day {letter['day']} · from {letter['from']}"
    if letter["days_since_previous"] is not None:
        line = line + f", {letter['days_since_previous']} days after the previous letter"
    line = line + f" · {len(letter['text'])} chars"
    return line

In [11]:
def get_recommendation() -> str:
    """
    Open the case: what NTSB recommended, who it was sent to, which letter you must review, NTSB's latest
    letter, the last grade NTSB gave in this case, and a list of all letters. Call this first.
    """
    c = current.case
    lines = []

    lines.append(f"CASE: {c['case_id']} | mode: {c['mode']} | priority: {c['priority']} | most wanted: {c['most_wanted']}")

    if c["recipient"]:
        lines.append(f"SENT TO: {c['recipient']}")
    else:
        lines.append("SENT TO: (name not in the data)")

    lines.append(f"RECOMMENDATION: {c['recommendation']}")
    lines.append(f"LETTER TO REVIEW: {c['to_evaluate']} (the newest letter, from the organisation)")

    # NTSB's latest letter: newest letter from NTSB
    latest_ntsb = None
    for letter in reversed(c["history"]):
        if letter["from"] == "NTSB":
            latest_ntsb = letter["letter_id"]
            break

    if latest_ntsb is not None:
        lines.append(f"NTSB'S LATEST LETTER: {latest_ntsb} (read it to see what NTSB last asked for)")
    else:
        lines.append("NTSB'S LATEST LETTER: none yet (the only request is the recommendation itself)")

    # last grade NTSB gave to OUR case
    grade, from_letter = last_ntsb_grade(c["history"], c["case_id"])
    if grade is not None:
        lines.append(f"LAST GRADE FROM NTSB IN THIS CASE: {grade} (in {from_letter})")
    else:
        lines.append("LAST GRADE FROM NTSB IN THIS CASE: none found in the letters")

    # letter list (no text)
    history = c["history"]
    lines.append(f"LETTERS ({len(history)}):")
    if len(history) <= 15:
        for letter in history:
            lines.append("  " + letter_line(letter))
    else:
        # long cases: first 3 + last 12, so the summary stays small
        for letter in history[:3]:
            lines.append("  " + letter_line(letter))
        lines.append(f"  ... {len(history) - 15} older letters not listed, use search_history ...")
        for letter in history[-12:]:
            lines.append("  " + letter_line(letter))

    return "\n".join(lines)


TOOLS["get_recommendation"] = get_recommendation

In [12]:
open_case(ferry)
print(run_tool("get_recommendation", {}))

CASE: CASE-7360 | mode: Marine | priority: CLASS II | most wanted: False
SENT TO: WV
RECOMMENDATION: TO THE 50 STATES:  Encourage your public ferry operators to voluntarily request application of the Federal requirements at 33 CFR 96 for implementing a safety management system, if they have not already done so.
LETTER TO REVIEW: L-0003 (the newest letter, from the organisation)
NTSB'S LATEST LETTER: L-0002 (read it to see what NTSB last asked for)
LAST GRADE FROM NTSB IN THIS CASE: Open – Acceptable (in L-0002)
LETTERS (3):
  L-0001 · Day 95 · from ORG · 462 chars
  L-0002 · Day 150 · from NTSB, 55 days after the previous letter · 679 chars
  L-0003 · Day 474 · from ORG, 324 days after the previous letter · 538 chars


**Batch-letter test:** a fake NTSB letter that grades our case **and** another one. The last grade must be ours.

In [13]:
fake = {
    "episode_id": "TEST-E1", "case_id": "CASE-0001", "mode": "Aviation", "priority": "CLASS II",
    "most_wanted": False, "recipient": "FAA", "recommendation": "Test recommendation.",
    "history": [
        {"letter_id": "L-0001", "day": 10, "days_since_previous": None, "from": "NTSB", "type": "",
         "text": 'CASE-0001 is classified "Open--Acceptable Response." [OTHER-REC] is classified "Closed--Unacceptable Action."',
         "_date": "x"},
        {"letter_id": "L-0002", "day": 200, "days_since_previous": 190, "from": "ORG", "type": "",
         "text": "We are working on it.", "_date": "x"},
    ],
    "to_evaluate": "L-0002", "answer": {}, "batch": True,
    "_srid": "TEST", "_reply_date": "2020-01-01", "_answer_date": "2020-02-01",
}
open_case(fake)
grade, from_letter = last_ntsb_grade(current.case["history"], current.case["case_id"])
print("last grade:", grade, "| should be: Open – Acceptable")

last grade: Open – Acceptable | should be: Open – Acceptable


## Tool 2 · `read_letter`
The AI saw the letter list. Now it reads one letter.

In the data, a letter is just one long string, no pages. Most letters fit easily (90% of replies are under
~3,000 characters), but a few are huge (up to 35,000), and that would fill the AI's whole desk at once. So
I cut a page on the fly: page 1 = characters 0–4,000, page 2 = 4,000–8,000, and so on. The header says
"page 1 of 3", so the AI knows there's more and can ask for page 2.

It also opens **peer letters** (other organisations' letters, from `get_peers` in tool 5), like `HI-L-0002`.
`find_letter` decides where to look: `L-0003` = our case, `HI-L-0002` = Hawaii's case (only after `get_peers`
has listed Hawaii). A wrong ID gets a friendly message back instead of a crash, so the AI can fix it.

In [14]:
def find_letter(letter_id):
    # our own letter: "L-0003"; a peer's letter: "HI-L-0002" (only after get_peers)
    letter_id = letter_id.strip().upper()

    if letter_id.startswith("L-"):
        return current.letter(letter_id)

    if "-L-" in letter_id:
        code, number = letter_id.split("-L-", 1)
        if code not in current.peers:
            return None
        for letter in peer_letters(current.peers[code]):
            if letter["letter_id"] == "L-" + number:
                return letter
    return None


def read_letter(letter_id: str, page: int = 1) -> str:
    """
    Read one letter, one page at a time. Your own case's letters look like "L-0003".
    Letters of other organisations (from get_peers) look like "HI-L-0002".

    Args:
        letter_id: The letter to read, like "L-0003" or "HI-L-0002".
        page: Which page to read, starting at 1. Long letters have more than one page.
    """
    h = find_letter(letter_id)
    if h is None:
        return (f"No letter called {letter_id}. Use get_recommendation for this case's letters, "
                "or get_peers first for other organisations' letters.")
    text = h["text"]

    # 1. how many pages (4,000 chars each, rounded up)
    pages = len(text) // PAGE_SIZE
    if len(text) % PAGE_SIZE != 0:
        pages = pages + 1
    if pages == 0:
        pages = 1

    # 2. page out of range
    if page < 1 or page > pages:
        return f"{letter_id} has {pages} page(s). Ask for page 1 to {pages}."

    # 3. this page's slice
    start = (page - 1) * PAGE_SIZE
    end = page * PAGE_SIZE
    chunk = text[start:end]

    # 4. header + slice
    header = f"{letter_id.strip().upper()} · Day {h['day']} · from {h['from']} · {len(text)} chars | page {page} of {pages}"
    return header + "\n\n" + chunk


TOOLS["read_letter"] = read_letter

In [15]:
open_case(ferry)
print(run_tool("read_letter", {"letter_id": "L-0003"}))
print(run_tool("read_letter", {"letter_id": "L-0009"}))          # no such letter
print(run_tool("read_letter", {"letter_id": "L-0003", "page": 2}))   # no page 2

L-0003 · Day 474 · from ORG · 538 chars | page 1 of 1

Letter Mail Controlled 4/20/[YEAR] 10:18:13 AM MC# 2060208 
4-20-05:  Email received from Mr. Donovan.
Mr. Brown, The WVPPA fully supports your recommendation included in Safety Recommendation CASE-7360. The Authority works in close cooperation with the Sistersville's ferry Board of Directors to ensure that the Sistersville Ferry on the Ohio River meets and maintains all State and National Maritime safety requirements. Sincere1y, Patrick J. Donovan, Acting Director, West Virginia Department of Transportation, Public Port Authority
No letter called L-0009. Use get_recommendation for this case's letters, or get_peers first for other organisations' letters.
L-0003 has 1 page(s). Ask for page 1 to 1.


## Tool 3 · `search_history`
Ctrl+F across this case's letters. For a long case (say 60 letters with the FAA about a rule), the AI searches
"final rule" and sees which letters talk about it, with a ~150-character preview, then reads only those.

It's plain keyword counting: score = how many times the words appear in each letter, best first, max 20.

**Does keyword search miss things?** Yes, if a letter says the same thing in other words ("the regulation will
be issued" vs "final rule"). Two things make that OK:
1. the docstring tells the AI to retry with other words if nothing comes back
2. I measured it: about **92% of episodes have a whole history under 16,000 characters**, so the AI can just
   read every letter. Search only matters for the ~8% of long cases

In [16]:
def hit_order(hit):
    # best score first; if two scores tie, the earlier letter first
    score = hit[0]
    position_in_case = hit[1]
    return (-score, position_in_case)


def search_history(query: str) -> str:
    """
    Search all letters in this case for words, and get short previews of where they appear.
    If nothing comes back, try other words with the same meaning (for example rule, regulation, rulemaking).

    Args:
        query: A few keywords, for example "final rule inspection".
    """
    # 1. the search words: lower-case, 3+ letters
    words = []
    for word in query.lower().split():
        word = word.strip(".,;:!?\"'()")
        if len(word) >= 3:
            words.append(word)
    if len(words) == 0:
        return "Give at least one keyword with 3 or more letters."

    # 2. score every letter = how many times the words appear in it
    hits = []
    position_in_case = 0
    for letter in current.case["history"]:
        text = letter["text"]
        text_lower = text.lower()

        score = 0
        first = -1
        for word in words:
            count = text_lower.count(word)
            score = score + count
            if count > 0:
                where = text_lower.find(word)
                if first == -1 or where < first:
                    first = where

        if score > 0:
            # preview: ~150 chars around the first hit
            start = first - 50
            if start < 0:
                start = 0
            preview = text[start:start + PREVIEW]
            preview = " ".join(preview.split())          # squash newlines and double spaces
            hits.append((score, position_in_case, letter, preview))

        position_in_case = position_in_case + 1

    # 3. nothing found
    if len(hits) == 0:
        return f"No letter mentions: {', '.join(words)}. Try other words with the same meaning."

    # 4. best first (if two scores tie, the earlier letter comes first)
    hits.sort(key=hit_order)

    # 5. top 20, one line each
    lines = [f"{len(hits)} letters match (showing up to {MAX_HITS}, best first):"]
    shown = 0
    for score, position_in_case, letter, preview in hits:
        if shown == MAX_HITS:
            break
        lines.append(f"{letter['letter_id']} (Day {letter['day']}, {letter['from']}, {score} hits): ...{preview}...")
        shown = shown + 1

    return "\n".join(lines)


TOOLS["search_history"] = search_history

In [17]:
open_case(ferry)
print(run_tool("search_history", {"query": "ferry safety management"}))
print(run_tool("search_history", {"query": "helicopter"}))
print(run_tool("search_history", {"query": "a of"}))

3 letters match (showing up to 20, best first):
L-0002 (Day 150, NTSB, 5 hits): ...The Safety Board understands that the West Virginia Department of Transportation (WVDOT) has assigned the task of implementing the recommended action...
L-0001 (Day 95, ORG, 4 hits): ...WVDOT takes an active interest in prevention and safety awareness for all modes of transportation in the state. The Public Port Authority, which is u...
L-0003 (Day 474, ORG, 4 hits): ...PA fully supports your recommendation included in Safety Recommendation CASE-7360. The Authority works in close cooperation with the Sistersville's fe...
No letter mentions: helicopter. Try other words with the same meaning.
Give at least one keyword with 3 or more letters.


## Tool 4 · `find_precedents` (the RAG tool)
The analyst's question: "how did NTSB grade replies like this one before?"

**The search index (built once):** every training case becomes one search document = its recommendation + the
reply NTSB graded. **BM25** scores how well a document's words match the query, and rare words ("ferry",
"SMS") count more than common ones ("safety"). It's keyword-based, not embeddings: it works on CPU in seconds,
it's easy to explain ("these words matched"), and NTSB letters reuse the same official terms a lot.
Embeddings are an "if time" upgrade.

**Two filters, so it can't cheat:**
- skip the **same recommendation**. On the ferry case, 8 of the top 12 raw hits were other states' replies to
  M-05-007, and the #1 hit was our own case
- skip anything NTSB decided **after** our reply date (the future)

**What the AI sees per past case:** ~220 characters of each part, to fit the desk (5 cases ≈ 3,300 characters):
- the start of the reply, with the "Letter Mail Controlled ... MC# 2030576" header cut off
- the **end** of NTSB's letter, because that's where the reason + the grade are

In [18]:
from rank_bm25 import BM25Okapi

STOP_WORDS = set("the and for that this with from are was were have has had will would been our your their its "
                 "which also into such than they them these those not but all any can may".split())

def tokens(text):
    # text -> list of search words (lower-case, 3+ letters, no common filler words)
    words = []
    for word in re.findall(r"[a-z0-9]+", text.lower()):
        if len(word) >= 3 and word not in STOP_WORDS:
            words.append(word)
    return words

# one search document per training episode: the recommendation + the reply NTSB graded
search_docs = []
for e in train:
    reply = e["history"][-1]["text"]
    search_docs.append(tokens(e["recommendation"] + " " + reply))

bm25 = BM25Okapi(search_docs)
print("precedent index:", len(search_docs), "training episodes")

precedent index: 20609 training episodes


In [19]:
MAIL_HEADER = re.compile(r"Letter Mail Controlled.*?MC#\s*\d+", re.I | re.S)

def short_start(text, n):
    # first n chars, after removing the "Letter Mail Controlled ... MC# 123" header
    text = MAIL_HEADER.sub("", text)
    text = " ".join(text.split())
    if text == "":
        return "(no text besides the mail header)"
    if len(text) <= n:
        return text
    return text[:n] + "..."

def short_end(text, n):
    # last n chars: NTSB's reason + grade are usually at the end
    text = " ".join(text.split())
    if len(text) <= n:
        return text
    return "..." + text[-n:]


def find_precedents(query: str, k: int = 5) -> str:
    """
    Find similar past cases and the grade NTSB gave them. Only cases NTSB decided before this reply are used.

    Args:
        query: What to search for, for example the main points of the reply you are reviewing.
        k: How many past cases to return, from 1 to 8.
    """
    # 1. keep k between 1 and 8
    k = int(k)
    if k < 1:
        k = 1
    if k > 8:
        k = 8

    # 2. score every training case against the query
    scores = bm25.get_scores(tokens(query))

    # 3. order the cases from best score to worst
    pairs = []
    for i in range(len(scores)):
        pairs.append((scores[i], i))
    pairs.sort(reverse=True)

    # 4. walk down the list, skip bad ones, stop at k
    picked = []
    for score, i in pairs:
        if score <= 0:
            break                                         # the rest don't match at all
        past = train[i]
        if past["_srid"] == current._srid:
            continue                                      # same recommendation -> would copy itself
        if past["_answer_date"] >= current._cutoff:
            continue                                      # decided after our reply -> the future
        picked.append(past)
        if len(picked) == k:
            break

    if len(picked) == 0:
        return "No earlier similar cases found."

    # 5. one block per past case
    lines = [f"{len(picked)} similar past cases (NTSB decided all of them before this reply):"]
    number = 1
    for past in picked:
        lines.append(f"{number}. {past['episode_id']} ({past['mode']}) | NTSB GRADE: {past['answer']['label']}")
        lines.append(f"   recommendation: {short_start(past['recommendation'], 160)}")
        lines.append(f"   reply: {short_start(past['history'][-1]['text'], 220)}")
        lines.append(f"   NTSB said: {short_end(past['answer']['ntsb_letter'], 220)}")
        number = number + 1

    return "\n".join(lines)


TOOLS["find_precedents"] = find_precedents

In [20]:
open_case(ferry)
print(run_tool("find_precedents", {"query": "fully supports recommendation encourage ferry operators safety management system", "k": 3}))

3 similar past cases (NTSB decided all of them before this reply):
1. CASE-6629-E1 (Marine) | NTSB GRADE: Open – Acceptable
   recommendation: TO THE U.S. COAST GUARD: Seek authority to require domestic towing companies to develop and implement an effective safety management system to ensure adequate m...
   reply: The Coast Guard believes that the use of safety management systems by all U.S. commercial vessels would result in significant benefits and supports the development of such programs. However, in order to justify a request...
   NTSB said: ...cordingly, Safety Recommendation CASE-6629 is classified "Open--Acceptable Response" pending confirmation from the Coast Guard that requirements for safety management systems will be promulgated for all towing companies.
2. CASE-3619-E1 (Railroad) | NTSB GRADE: Closed – Acceptable
   recommendation: The NTSB recommends that the tourist railway association, inc.: encourage its members who operate steam locomotives to cooperate with the FRA

## Tool 5 · `get_peers`
The same recommendation often goes to many organisations (M-05-007 went to 42 states). Before grading West
Virginia, an analyst checks how NTSB handled **the other states** on the same ask.

My design:
1. find the other organisations that got this same recommendation, with a grade **before our reply date**
2. count their grades ("10 × Closed – Acceptable, 5 × Closed – Other")
3. show the **5 most recent** peers, each with their letters (IDs like `HI-L-0002`), the grade at every NTSB
   letter, and two previews (the start of their reply, the end of NTSB's answer) so the AI can see who's
   similar before opening anything
4. the AI can open any peer letter with `read_letter("HI-L-0002")`

`by_srid` groups every episode by recommendation. Using all three piles is safe: in `02` I split by
recommendation, so all episodes of one recommendation sit in the same pile anyway.

In [21]:
# all episodes of all piles, grouped by recommendation (real ID)
by_srid = {}
for e in train + val + test:
    srid = e["_srid"]
    if srid not in by_srid:
        by_srid[srid] = []
    by_srid[srid].append(e)
print(len(by_srid), "recommendations grouped")

8367 recommendations grouped


In [22]:
from datetime import date

MAX_PEERS = 5

def days_between(a, b):
    # "2006-04-20", "2006-07-06" -> 77
    return (date.fromisoformat(b[:10]) - date.fromisoformat(a[:10])).days


def peer_letters(past):
    # a peer's letters: its history + NTSB's answer to it (the grade letter), all before our date
    letters = []
    for letter in past["history"]:
        letters.append(letter)

    reply = past["history"][-1]
    gap = days_between(past["_reply_date"], past["_answer_date"])
    answer_letter = {
        "letter_id": f"L-{len(past['history']) + 1:04d}",
        "day": reply["day"] + gap,
        "days_since_previous": gap,
        "from": "NTSB",
        "text": past["answer"]["ntsb_letter"],
    }
    letters.append(answer_letter)
    return letters


def get_peers() -> str:
    """
    See how NTSB handled the other organisations that got this same recommendation, before this reply:
    their letters and the grade NTSB gave at each step. Read a peer letter with read_letter, for example "HI-L-0002".
    """
    # 1. latest episode of every OTHER organisation, answered before our reply date
    latest = {}
    unknown = 0
    for past in by_srid[current._srid]:
        if past["recipient"] == current.case["recipient"]:
            continue                              # skip our own organisation
        if past["_answer_date"] >= current._cutoff:
            continue                              # decided after our reply -> the future

        code = past["recipient"]
        if code == "":
            unknown = unknown + 1
            code = f"ORG{unknown}"                # name not in the data

        if code not in latest:
            latest[code] = past
        elif past["_answer_date"] > latest[code]["_answer_date"]:
            latest[code] = past

    if len(latest) == 0:
        return "No other organisation got a grade for this recommendation before this reply."

    # 2. count the grades of all peers
    counts = {}
    for code in latest:
        label = latest[code]["answer"]["label"]
        if label not in counts:
            counts[label] = 0
        counts[label] = counts[label] + 1
    count_texts = []
    for label in counts:
        count_texts.append(f"{counts[label]} × {label}")

    # 3. the most recent peers first, keep 5
    order = []
    for code in latest:
        order.append((latest[code]["_answer_date"], code))
    order.sort(reverse=True)

    current.peers = {}
    lines = [f"{len(latest)} other organisations got this recommendation. Latest grades before this reply: " + ", ".join(count_texts)]
    lines.append(f"The {min(MAX_PEERS, len(order))} most recent (read any letter with read_letter, e.g. \"{order[0][1]}-L-0001\"):")

    shown = 0
    for answered, code in order:
        if shown == MAX_PEERS:
            break
        past = latest[code]
        current.peers[code] = past
        letters = peer_letters(past)
        lines.append(f"{code} · {len(letters)} letters")
        for letter in letters:
            line = f"   {code}-{letter['letter_id']} · Day {letter['day']} · {letter['from']}"
            if letter["from"] == "NTSB":
                grades = grades_for_case(letter["text"], current.case["case_id"])
                if len(grades) > 0:
                    line = line + f" → {grades[-1]}"
            lines.append(line)

        # previews, so the AI can decide which letter is worth opening
        last_reply = past["history"][-1]
        lines.append(f"   their reply: {short_start(last_reply['text'], 120)}")
        lines.append(f"   NTSB said: {short_end(past['answer']['ntsb_letter'], 150)}")
        shown = shown + 1

    return "\n".join(lines)


TOOLS["get_peers"] = get_peers

In [23]:
open_case(ferry)
print(run_tool("read_letter", {"letter_id": "HI-L-0002"}))      # before get_peers -> not found
print(run_tool("get_peers", {}))
print(run_tool("read_letter", {"letter_id": "HI-L-0002"}))      # now it opens Hawaii's NTSB letter

No letter called HI-L-0002. Use get_recommendation for this case's letters, or get_peers first for other organisations' letters.
15 other organisations got this recommendation. Latest grades before this reply: 10 × Closed – Acceptable, 5 × Closed – Other
The 5 most recent (read any letter with read_letter, e.g. "HI-L-0001"):
HI · 2 letters
   HI-L-0001 · Day 222 · ORG
   HI-L-0002 · Day 269 · NTSB → Closed – Acceptable
   their reply: Thank you for your fax dated March 18, [YEAR] regarding the National Transportation Safety Board's (NTSB) recommendation...
   NTSB said: ...sed ferries in the State of the content of the recommendation. Accordingly, Safety Recommendation CASE-7360 is classified "Closed--Acceptable Action."
NJ · 2 letters
   NJ-L-0001 · Day 191 · ORG
   NJ-L-0002 · Day 249 · NTSB → Closed – Acceptable
   their reply: Following your letter and recommendation, we have forwarded information regarding 33 CFR Part 96 (Rules for the Safe Ope...
   NTSB said: ...e level of inter

## Tool 6 · `verify_quote`
The AI has to back up its grade with a quote, like *"The WVPPA fully supports your recommendation"*. LLMs
sometimes make up quotes that sound right. If an analyst trusted a fake one, NTSB could close a safety
recommendation that isn't actually done. So plain code (not the AI) checks the quote is really there, word for
word, ignoring capitals and extra spaces.

A quote must be **at least 4 words**: "the Board" or "we agree" appear in almost every letter, so they'd
"verify" without proving anything.

`check_rules` uses this, and a fake quote blocks the draft.

In [24]:
def verify_quote(letter_id: str, quote: str) -> str:
    """
    Check that a quote really appears, word for word, in one letter. Use it before you rely on a quote.

    Args:
        letter_id: The letter the quote comes from, like "L-0003" or "HI-L-0002".
        quote: The exact words being quoted from that letter.
    """
    # 1. find the letter
    h = find_letter(letter_id)
    if h is None:
        return f"NOT FOUND: there is no letter {letter_id}."

    # 2. clean both the same way: lower-case, one space between words
    clean_quote = " ".join(quote.lower().split())
    clean_text = " ".join(h["text"].lower().split())

    # 3. a tiny quote proves nothing
    if len(clean_quote.split()) < 4:
        return "NOT FOUND: the quote is too short to count as evidence (use at least 4 words)."

    # 4. is it really there?
    if clean_quote in clean_text:
        return f"VERIFIED: the quote is in {letter_id}."
    return f"NOT FOUND: {letter_id} does not contain these exact words."


TOOLS["verify_quote"] = verify_quote

In [25]:
open_case(ferry)
print(run_tool("verify_quote", {"letter_id": "L-0003", "quote": "The WVPPA  fully supports your\nrecommendation"}))
print(run_tool("verify_quote", {"letter_id": "L-0003", "quote": "the WVPPA has closed the ferry"}))
print(run_tool("verify_quote", {"letter_id": "L-0009", "quote": "fully supports your recommendation"}))
print(run_tool("verify_quote", {"letter_id": "L-0003", "quote": "the"}))

VERIFIED: the quote is in L-0003.
NOT FOUND: L-0003 does not contain these exact words.
NOT FOUND: there is no letter L-0009.
NOT FOUND: the quote is too short to count as evidence (use at least 4 words).


## Helper 7 · `days_late`
NTSB asks every organisation to reply within **90 days**. This measures how long the org took to answer
**NTSB's latest letter**, and returns the days past those 90 (never below 0).

- found an org letter after it → measure to that letter
- none yet → measure to `today` (in live use: "overdue, send a reminder", like the 755 silent cases from the audit)
- **if NTSB's latest letter closed the case, it returns 0**: there's nothing left to answer, so a closed case
  must never be flagged as overdue (I caught this one)

It's not a tool the AI calls. `check_rules` uses it to add a WARN, so the human sees it and the letter can
mention it. It doesn't change the grade by itself.

In [26]:
def days_late(history, today, case_id):
    # 1. position of NTSB's latest letter
    ntsb_position = -1
    position = 0
    for letter in history:
        if letter["from"] == "NTSB":
            ntsb_position = position
        position = position + 1
    if ntsb_position == -1:
        return 0                                   # NTSB never wrote -> nothing to be late for

    ntsb_letter = history[ntsb_position]

    # 2. if that letter CLOSED our case, there's nothing left to answer
    grades = grades_for_case(ntsb_letter["text"], case_id)
    if len(grades) > 0 and grades[-1].startswith("Closed"):
        return 0

    # 3. first org letter after it (or today, if they haven't answered yet)
    end_day = today
    for letter in history[ntsb_position + 1:]:
        if letter["from"] == "ORG":
            end_day = letter["day"]
            break

    # 4. days past the 90-day window
    late = end_day - ntsb_letter["day"] - REPLY_WINDOW
    if late < 0:
        late = 0
    return late

In [27]:
def H(letter_id, day, sender, text=""):
    return {"letter_id": letter_id, "day": day, "from": sender, "text": text}

print(days_late([H("L-1", 0, "NTSB"), H("L-2", 60, "ORG")], 60, "CASE-1"), "want 0")        # on time
print(days_late([H("L-1", 0, "NTSB"), H("L-2", 400, "ORG")], 400, "CASE-1"), "want 310")    # late
print(days_late([H("L-1", 0, "ORG"), H("L-2", 50, "NTSB"), H("L-3", 100, "ORG")], 100, "CASE-1"), "want 0")
print(days_late([H("L-1", 0, "NTSB")], 1000, "CASE-1"), "want 910")                          # no answer yet
print(days_late([H("L-1", 0, "ORG")], 500, "CASE-1"), "want 0")                              # no NTSB letter
print(days_late([H("L-1", 0, "NTSB", 'CASE-1 is classified "Closed--Acceptable Action."')], 1000, "CASE-1"), "want 0")   # closed

open_case(ferry)
print(days_late(current.case["history"], current.case["history"][-1]["day"], current.case["case_id"]), "ferry, want 234")

0 want 0
310 want 310
0 want 0
910 want 910
0 want 0
0 want 0
234 ferry, want 234


## Tools 8–9 · Draft → rules → approval → human
1. **`submit_review`**: the AI hands in its draft (grade, confidence, reasons, one quote, the NTSB letter).
   It's saved, **nothing is sent**
2. **`check_rules`**: our code runs the rulebook. Any FAIL = not ready:
   - the grade is one of our 7
   - the quote passes `verify_quote`
   - confidence is 0–1; under 0.6 → WARN (a human should look closer)
   - the letter names the case and stays under ~1,900 characters
   - late reply → WARN (from `days_late`)
   - who must approve: an analyst for Open grades, **analyst + supervisor for any Closed grade**
3. **`request_approval`**: puts the draft in a human's queue, **only if the rules passed**
4. **`record_decision`**: the human's approve/reject click. It's **not in `TOOLS`**, so the AI can't approve its
   own work. A rejection is logged with the note; sending it back to the AI to revise goes in the agent loop
   (next notebook), and the demo gets Approve / Reject / Edit buttons
5. **`read_attachment`**: our data has no attachment files, only mentions like "enclosed". So it says so honestly
   and flags it for the analyst. Faking an attachment would be lying

In [28]:
def submit_review(grade: str, confidence: float, reasons: str, quote_letter_id: str, quote: str, letter: str) -> str:
    """
    Hand in your review draft. Nothing is sent: it is saved for the rule check and human approval.

    Args:
        grade: One of: Open – Acceptable, Open – Acceptable Alternate, Open – Unacceptable, Closed – Acceptable, Closed – Acceptable Alternate, Closed – Unacceptable, Closed – Other.
        confidence: How sure you are about the grade, from 0.0 to 1.0.
        reasons: Two to four sentences on why this grade fits.
        quote_letter_id: The letter your evidence quote comes from, like "L-0003".
        quote: Exact words from that letter (at least 4 words) that support the grade.
        letter: The draft NTSB letter to the organisation, ending with the classification.
    """
    # the AI may type "-" instead of "–": make it match our 7 names
    clean_grade = grade.strip().replace(" - ", " – ")

    current.draft = {
        "grade": clean_grade,
        "confidence": confidence,
        "reasons": reasons,
        "quote_letter_id": quote_letter_id,
        "quote": quote,
        "letter": letter,
    }
    current.rules_passed = False                  # a new draft must be checked again
    current.log.append({"step": len(current.log), "time": now(), "event": "draft saved", "draft": current.draft})
    return "Draft saved. Next: call check_rules."


def check_rules() -> str:
    """
    Run the rule check on your saved draft. Fix anything marked FAIL and submit again before asking for approval.
    """
    d = current.draft
    if d is None:
        return "No draft yet. Call submit_review first."

    results = []          # list of (status, message)

    # rule 1: the grade must be one of our 7
    if d["grade"] in GRADES:
        results.append(("PASS", f"grade is one of the 7 grades ({d['grade']})"))
    else:
        results.append(("FAIL", f"'{d['grade']}' is not one of the 7 grades"))

    # rule 2: the quote must really be in the letter
    check = verify_quote(d["quote_letter_id"], d["quote"])
    if check.startswith("VERIFIED"):
        results.append(("PASS", "evidence quote: " + check))
    else:
        results.append(("FAIL", "evidence quote: " + check))

    # rule 3: confidence is a number from 0 to 1; below 0.6 a human should look closer
    try:
        confidence = float(d["confidence"])
    except (TypeError, ValueError):
        confidence = -1
    if confidence < 0 or confidence > 1:
        results.append(("FAIL", "confidence must be a number from 0 to 1"))
    elif confidence < LOW_CONFIDENCE:
        results.append(("WARN", f"low confidence ({confidence}): a human should look closely"))
    else:
        results.append(("PASS", f"confidence {confidence}"))

    # rule 4: the letter names the case and isn't too long
    if current.case["case_id"] in d["letter"]:
        results.append(("PASS", "letter names the case ID"))
    else:
        results.append(("FAIL", f"letter must name the case ID ({current.case['case_id']})"))
    if len(d["letter"]) <= LETTER_LIMIT:
        results.append(("PASS", f"letter length {len(d['letter'])} chars (limit {LETTER_LIMIT})"))
    else:
        results.append(("FAIL", f"letter is {len(d['letter'])} chars, limit is {LETTER_LIMIT}"))

    # rule 5: late reply -> warn, so the letter can mention it
    history = current.case["history"]
    today = history[-1]["day"]
    late = days_late(history, today, current.case["case_id"])
    if late > 0:
        results.append(("WARN", f"reply came {late} days after the 90-day window"))
    else:
        results.append(("PASS", "reply came within 90 days"))

    # rule 6: who must approve
    if d["grade"].startswith("Closed"):
        results.append(("INFO", "approval needed from: analyst + supervisor"))
    else:
        results.append(("INFO", "approval needed from: analyst"))

    # passed = no FAIL anywhere
    passed = True
    for status, message in results:
        if status == "FAIL":
            passed = False
    current.rules_passed = passed
    current.log.append({"step": len(current.log), "time": now(), "event": "rules checked",
                        "results": results, "passed": passed})

    lines = []
    for status, message in results:
        lines.append(f"{status}: {message}")
    if passed:
        lines.append("READY for approval.")
    else:
        lines.append("NOT READY: fix the FAIL items and submit again.")
    return "\n".join(lines)


def request_approval() -> str:
    """
    Send your checked draft to the human reviewers. Only works after check_rules says READY.
    """
    if current.draft is None:
        return "Refused: no draft. Call submit_review first."
    if not current.rules_passed:
        return "Refused: the draft has not passed check_rules."

    if current.draft["grade"].startswith("Closed"):
        level = "analyst + supervisor"
    else:
        level = "analyst"
    current.approval = {"level": level, "status": "pending", "decisions": []}
    current.log.append({"step": len(current.log), "time": now(), "event": "approval requested", "level": level})
    return f"Sent for approval to: {level}. Status: pending. Your part is done."


def read_attachment(name: str) -> str:
    """
    Try to open an attachment mentioned in a letter.

    Args:
        name: What the letter calls the attachment, for example "enclosed report".
    """
    current.log.append({"step": len(current.log), "time": now(), "event": "attachment flagged", "name": name})
    return (f"Attachment '{name}' is not available in this system. Flagged for the analyst to check by hand. "
            "Do not guess what it says.")


TOOLS["submit_review"] = submit_review
TOOLS["check_rules"] = check_rules
TOOLS["request_approval"] = request_approval
TOOLS["read_attachment"] = read_attachment


def record_decision(role, approved, note=""):
    # the human's click. NOT a tool: it's not in TOOLS, so the AI can't call it
    current.approval["decisions"].append({"role": role, "approved": approved, "note": note, "time": now()})

    if current.approval["level"] == "analyst + supervisor":
        needed = ["analyst", "supervisor"]
    else:
        needed = ["analyst"]

    said_yes = []
    for decision in current.approval["decisions"]:
        if decision["approved"]:
            said_yes.append(decision["role"])

    if not approved:
        current.approval["status"] = "rejected"
    else:
        everyone = True
        for role_needed in needed:
            if role_needed not in said_yes:
                everyone = False
        if everyone:
            current.approval["status"] = "approved"

    current.log.append({"step": len(current.log), "time": now(), "event": "human decision",
                        "role": role, "approved": approved, "note": note, "status": current.approval["status"]})
    return current.approval["status"]

In [29]:
open_case(ferry)
letter = ("Thank you for your letter about the Sistersville Ferry. We note that the WVPPA fully supports the "
          "recommendation and works with the ferry's Board of Directors to meet State and national maritime "
          "safety requirements. Accordingly, Safety Recommendation CASE-7360 is classified \"Closed—Acceptable Action.\"")

# 1. a made-up quote -> FAIL, approval refused
print(run_tool("submit_review", {"grade": "Closed - Acceptable", "confidence": 0.8, "reasons": "test",
                                 "quote_letter_id": "L-0003", "quote": "the WVPPA has closed the ferry", "letter": letter}))
print(run_tool("check_rules", {}))
print(run_tool("request_approval", {}))
print("---")

# 2. a real quote -> READY -> approval
print(run_tool("submit_review", {"grade": "Closed - Acceptable", "confidence": 0.8,
                                 "reasons": "Most peers were closed acceptable; WV supports and cooperates.",
                                 "quote_letter_id": "L-0003", "quote": "The WVPPA fully supports your recommendation",
                                 "letter": letter}))
print(run_tool("check_rules", {}))
print(run_tool("request_approval", {}))
print("---")

# 3. the humans
print("analyst ->", record_decision("analyst", True, "looks right"))
print("supervisor ->", record_decision("supervisor", True, "agree"))
print("NTSB's real grade:", ferry["answer"]["label"])
print("---")

# 4. the AI can't approve itself
print(run_tool("record_decision", {"role": "supervisor", "approved": True}))

Draft saved. Next: call check_rules.
PASS: grade is one of the 7 grades (Closed – Acceptable)
FAIL: evidence quote: NOT FOUND: L-0003 does not contain these exact words.
PASS: confidence 0.8
PASS: letter names the case ID
PASS: letter length 302 chars (limit 1900)
WARN: reply came 234 days after the 90-day window
INFO: approval needed from: analyst + supervisor
NOT READY: fix the FAIL items and submit again.
Refused: the draft has not passed check_rules.
---
Draft saved. Next: call check_rules.
PASS: grade is one of the 7 grades (Closed – Acceptable)
PASS: evidence quote: VERIFIED: the quote is in L-0003.
PASS: confidence 0.8
PASS: letter names the case ID
PASS: letter length 302 chars (limit 1900)
WARN: reply came 234 days after the 90-day window
INFO: approval needed from: analyst + supervisor
READY for approval.
Sent for approval to: analyst + supervisor. Status: pending. Your part is done.
---
analyst -> pending
supervisor -> approved
NTSB's real grade: Closed – Acceptable
---
ERRO

### The audit log of that run
Everything above, written down by `run_tool` and the decision tools, and saved to Drive.

In [30]:
for entry in current.log:
    what = entry.get("tool")
    if what is None:
        what = entry.get("event")
    print(entry["step"], entry["time"], what)
print("saved to:", save_log(current))

0 2026-11-14T16:37:28+00:00 case opened
1 2026-11-14T16:37:28+00:00 draft saved
2 2026-11-14T16:37:28+00:00 submit_review
3 2026-11-14T16:37:28+00:00 rules checked
4 2026-11-14T16:37:28+00:00 check_rules
5 2026-11-14T16:37:28+00:00 request_approval
6 2026-11-14T16:37:28+00:00 draft saved
7 2026-11-14T16:37:28+00:00 submit_review
8 2026-11-14T16:37:28+00:00 rules checked
9 2026-11-14T16:37:28+00:00 check_rules
10 2026-11-14T16:37:28+00:00 approval requested
11 2026-11-14T16:37:28+00:00 request_approval
12 2026-11-14T16:37:28+00:00 human decision
13 2026-11-14T16:37:28+00:00 human decision
14 2026-11-14T16:37:28+00:00 record_decision
saved to: /content/drive/MyDrive/ntsb_agent/logs/CASE-7360-E11.jsonl


## The tool menu (what Tiny Aya will read)
`get_json_schema` turns each tool's docstring + inputs into the JSON menu that goes into Tiny Aya's system
prompt (the fallback format from `00`, because Tiny Aya's chat template drops `tools=`). If a docstring is
broken, this cell errors, so it's a free check. The `#` comments inside the code never reach the model.

In [31]:
from transformers.utils import get_json_schema

TOOL_SCHEMAS = []
for name in TOOLS:
    TOOL_SCHEMAS.append(get_json_schema(TOOLS[name]))

for schema in TOOL_SCHEMAS:
    function = schema["function"]
    params = []
    for p in function["parameters"].get("properties", {}):
        params.append(p)
    print(f"{function['name']}({', '.join(params)}): {function['description'][:80]}")

menu_text = json.dumps(TOOL_SCHEMAS)
print()
print(len(TOOL_SCHEMAS), "tools | menu size:", len(menu_text), "chars (about", len(menu_text) // 4, "tokens)")

get_recommendation(): Open the case: what NTSB recommended, who it was sent to, which letter you must 
read_letter(letter_id, page): Read one letter, one page at a time. Your own case's letters look like "L-0003".
search_history(query): Search all letters in this case for words, and get short previews of where they 
find_precedents(query, k): Find similar past cases and the grade NTSB gave them. Only cases NTSB decided be
get_peers(): See how NTSB handled the other organisations that got this same recommendation, 
verify_quote(letter_id, quote): Check that a quote really appears, word for word, in one letter. Use it before y
submit_review(grade, confidence, reasons, quote_letter_id, quote, letter): Hand in your review draft. Nothing is sent: it is saved for the rule check and h
check_rules(): Run the rule check on your saved draft. Fix anything marked FAIL and submit agai
request_approval(): Send your checked draft to the human reviewers. Only works after check_rules say
read_attachmen

## Safety check on 50 real cases
I run the reading tools on 50 validation cases and check:
1. NTSB's sealed answer letter **never** shows up in any tool output
2. every precedent is from the past and from another recommendation

In [32]:
import random
random.seed(7)

problems = []
checked = 0
for e in random.sample(val, 50):
    open_case(e)
    reply_id = current.case["to_evaluate"]
    reply_text = current.letter(reply_id)["text"]

    outputs = []
    outputs.append(run_tool("get_recommendation", {}))
    outputs.append(run_tool("read_letter", {"letter_id": reply_id}))
    outputs.append(run_tool("search_history", {"query": reply_text[:80]}))
    outputs.append(run_tool("get_peers", {}))
    precedents = run_tool("find_precedents", {"query": reply_text[:600], "k": 8})
    outputs.append(precedents)

    # 1. the sealed answer letter must never show up in any tool output
    secret = " ".join(e["answer"]["ntsb_letter"].split())[:200]
    for out in outputs:
        if secret != "" and secret in " ".join(out.split()):
            problems.append(("answer leaked", e["episode_id"]))

    # 2. every precedent must be from the past and from another recommendation
    for line in precedents.split("\n"):
        match = re.match(r"\d+\. (CASE-\d+-E\d+) ", line)
        if match:
            for past in train:
                if past["episode_id"] == match.group(1):
                    if past["_answer_date"] >= e["_reply_date"]:
                        problems.append(("precedent from the future", e["episode_id"], past["episode_id"]))
                    if past["_srid"] == e["_srid"]:
                        problems.append(("precedent from the same recommendation", e["episode_id"], past["episode_id"]))
                    break
    checked = checked + 1

print(checked, "cases checked |", len(problems), "problems")
for p in problems[:10]:
    print("  ", p)

50 cases checked | 0 problems


## Where I landed
- **Desk:** one case at a time, visible part only. No tool can reach the answer
- **10 tools:** get_recommendation, read_letter (+ peer letters), search_history, find_precedents, get_peers,
  verify_quote, submit_review, check_rules, request_approval, read_attachment. Plus `days_late` (helper) and
  `record_decision` (human only)
- **Rules + approval:** fake quotes blocked, Closed needs analyst + supervisor, the AI can't approve itself
- **Audit log:** every step, saved per case to Drive
- **Checked:** the ferry case end to end, and 0 problems on 50 validation cases

**Decisions I made along the way:**
- dropped `get_outstanding_asks` (keyword "meaning" was fake) → point to NTSB's latest letter instead
- fixed the batch-letter last-grade bug (7.5% of episodes)
- peers show their letters + previews, and their letters are readable
- `days_late` is 0 when the case is already closed
- quotes need 4+ words
- keyword search is enough for now (~92% of cases fit on the desk); embeddings only if time

**Next (`04`, GPU):** the agent loop that connects Tiny Aya to these tools, with a desk meter, and the "before" score.